In [ ]:
import os
from dotenv import load_dotenv,find_dotenv
_= load_dotenv(find_dotenv())
groq_api_key = os.environ["GROQ_API_KEY"]

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='llama-3.3-70b-versatile')

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(

    messages=[
        ("tell me short story about {player}")
    ]
)

output_parser = StrOutputParser()

chain = prompt | llm | output_parser


In [ ]:
chain.invoke({"player":"ronaldo"})

## Runnables Execution Alternatives 
* Runnables are like prompt llm outputparser
* Execution alternatives are - invoke,stream,batch

Stream - streaming response like chatgpt

In [ ]:
for s in chain.stream({"player":"ronaldo"}):
    print(s,end="",flush=True)

Batch - more than one input

In [ ]:
chain.batch([{"player":"ronaldo"},{"player":"messi"}])

## Built in Runnables

In [ ]:
import os 
from dotenv import load_dotenv,find_dotenv
_= load_dotenv(find_dotenv())
groq_api_key = os.environ['GROQ_API_KEY']

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='llama-3.3-70b-versatile')

## 1. Runnable Passthrough
* It does not do anything to the input data 
* input = output

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough()

In [ ]:
chain.invoke('hey')

## Runnable lamda
* To use a custom function inside a LCEL chain we need to wrap it up with RunnableLambda.
* lets define a simple function to demostrate the use 

In [ ]:
def russian_surname(name: str)-> str:
    return f'{name}ovich'

In [ ]:
from langchain_core.runnables import RunnableLambda

chain = RunnablePassthrough() | RunnableLambda(russian_surname)

chain.invoke("pratik")

## Runnable Parallel
* we will use runnableparallel() for running the tasks in parallel
* this is the most important and most usefull runnable from langchain

In [37]:
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel(
{
    "operation a": RunnablePassthrough(),
    "operation b": RunnableLambda(russian_surname)

}
)

chain.invoke("pratik")

{'operation a': 'pratik', 'operation b': 'pratikovich'}

## Advanced example of runnable passthrough

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [59]:
llm = ChatGroq(model='llama-3.3-70b-versatile')
vector_store = FAISS.from_texts(["im john json aka jj im a professional painter providing a services for painting homes im a very positive person"],embedding=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2'))

In [46]:
retriver = vector_store.as_retriever()

template = """ answer the question based on the following context 

{context}

Question : {question}

"""

prompt = ChatPromptTemplate.from_template(template)


retrival_chain = (
        RunnableParallel({"context": retriver , "question": RunnablePassthrough()}) 
        | prompt
        | llm
        |StrOutputParser()
        )

retrival_chain.invoke('who is jj')


'According to the given context, JJ is John Json, a professional painter who provides painting services for homes.'

## itemgeter
* when we want to provide multiple inputs to the model we use itemgeter

In [50]:
from operator import itemgetter
retriver = vector_store.as_retriever()

template = """ answer the question based on the following context 

{context}

Question : {question}
answer in the followinglanguage : {language}
"""

prompt = ChatPromptTemplate.from_template(template)


retrival_chain = (
        {
            "context": itemgetter("question")|retriver,
            "question": itemgetter("question"),
            "language": itemgetter("language")
         
        } # Runnable parallel syntax { }
        | prompt
        | llm
        |StrOutputParser()
        )

retrival_chain.invoke({"question":'who is jj',"language":"hindi"})


'जेजे जॉन जेसन हैं, जो एक पेशेवर पेंटर हैं और घरों की पेंटिंग की सेवाएं प्रदान करते हैं।'

## Inbuilt functions inside runnables
* .bind()
* for example we can add an argument to stop the model response when it reaches the word ronaldo 

In [56]:
retrival_chain = (
        {
            "context": itemgetter("question")|retriver,
            "question": itemgetter("question"),
            "language": itemgetter("language")
         
        } # Runnable parallel syntax { }
        | prompt
        | llm.bind(stop=['JSON'])
        |StrOutputParser()
        )

retrival_chain.invoke({"question":'who is jj',"language":"english"})

'JJ is John '

## Combined chain

In [62]:
retriver = vector_store.as_retriever()

template = """ answer the question based on the following context 

{context}

Question : {question}

"""

prompt = ChatPromptTemplate.from_template(template)


retrival_chain = (
        RunnableParallel({"context": retriver , "question": RunnablePassthrough()}) 
        | prompt
        | llm
        |StrOutputParser()
        )

retrival_chain.invoke('who is jj')


'According to the context, JJ (also known as John JSON) is a professional painter who provides painting services for homes. He describes himself as a very positive person.'

In [68]:
historian_prompt = ChatPromptTemplate.from_template(" was {bias} positive or negative person?")

composed_chain = {"bias":retrival_chain} | historian_prompt | llm | StrOutputParser()

composed_chain.invoke("jj")

"Based on the context provided, JJ (John JSON) appears to be a positive person. The fact that he is a professional painter who provides painting services for homes suggests that he is a skilled and helpful individual who contributes to making people's homes look better. There is no negative information provided about him, so it can be inferred that he is a positive person."